> **Notebook Goal**\n> Train a KNN income classifier, measure global individual fairness (IF), then report within-group IF for marital-status, sex, and their intersection.


> **Imports**\n> Load pandas and scikit-learn components for preprocessing, KNN training, and evaluation metrics.


In [8]:
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler


> **Targets and Sensitive Attributes**\n> Define the prediction target and list sensitive attributes. Sensitive attributes are excluded from training features and kept for fairness analysis.


In [9]:
target = "income-class"
sensitive_cols = ["marital-status", "relationship", "race", "sex", "native-country"]

> **Load and Split Data**\n> Read the Adult dataset, separate train/test sets, and preserve sensitive columns for later subgroup analysis.


In [10]:
data = pd.read_csv("adult.data.csv", na_values="?", skipinitialspace=True)

feature_cols = [c for c in data.columns if c not in [target] + sensitive_cols]
X = data[feature_cols]
y = data[target]
X_sensitive = data[sensitive_cols]

X_train, X_test, y_train, y_test, _, X_sensitivefeatures_test = train_test_split(
    X, y, X_sensitive, test_size=0.20, random_state=42, stratify=y
)

> **Preprocessing Pipeline**\n> Create numeric and categorical preprocessing steps (impute + scale / one-hot) used by the KNN pipeline.


In [11]:
numeric_cols = X_train.select_dtypes(include=["number"]).columns.tolist()
categorical_cols = X_train.select_dtypes(exclude=["number"]).columns.tolist()

numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore")),
])

preprocess = ColumnTransformer([
    ("num", numeric_pipeline, numeric_cols),
    ("cat", categorical_pipeline, categorical_cols),
])

> **Train Baseline KNN**\n> Fit the selected KNN configuration and print standard classification metrics before fairness analysis.


In [12]:
classifier = KNeighborsClassifier(
    n_neighbors=17,
    weights="uniform",
    leaf_size=20,
    p=2,
)

classifierPipeline = Pipeline([
    ("preprocess", preprocess),
    ("knn", classifier),
])

classifierPipeline.fit(X_train, y_train)
y_predict = classifierPipeline.predict(X_test)  # <- use pipeline here

print(confusion_matrix(y_test, y_predict))
print(classification_report(y_test, y_predict))

[[4615  330]
 [ 860  708]]
              precision    recall  f1-score   support

       <=50K       0.84      0.93      0.89      4945
        >50K       0.68      0.45      0.54      1568

    accuracy                           0.82      6513
   macro avg       0.76      0.69      0.71      6513
weighted avg       0.80      0.82      0.80      6513



## Individual Fairness Analysis (Minimal)
Concise IF audit using local nearest-neighbor pairs, probability outputs, and a normalized distance metric in preprocessed non-sensitive feature space.


> **Global IF (Local Pair Audit)**\n> Compute local pairs, normalize distances, calculate violations `max(0, |p_i-p_j| - d_ij)`, and print the global IF summary.


In [13]:
import numpy as np
from sklearn.neighbors import NearestNeighbors

# Probability output p(x) for the positive class
p = classifierPipeline.predict_proba(X_test)[:, list(classifierPipeline.classes_).index(">50K")]

# Similarity metric d(x, y): normalized distance in preprocessed non-sensitive feature space
X_test_preprocessed = classifierPipeline.named_steps["preprocess"].transform(X_test)

# Local pairs: fixed number of nearest neighbors (excluding self)
k_local = 15
nn = NearestNeighbors(n_neighbors=min(k_local + 1, len(X_test)), metric="euclidean")
nn.fit(X_test_preprocessed)
distances, indices = nn.kneighbors(X_test_preprocessed)

rows = np.repeat(np.arange(len(X_test)), distances.shape[1] - 1)
cols = indices[:, 1:].reshape(-1)
raw_d = distances[:, 1:].reshape(-1)

# Normalize distances into [0, 1]
scale = raw_d.max()
if scale == 0:
    scale = 1.0
d = np.clip(raw_d / scale, 0.0, 1.0)

# Fairness violations: max(0, |p_i - p_j| - d_ij)
delta_p = np.abs(p[rows] - p[cols])
violations = np.maximum(0.0, delta_p - d)

print("Individual fairness (local) summary")
print("- evaluated pairs:", len(violations))
print("- max_violation:", round(float(violations.max()), 6))
print("- avg_violation:", round(float(violations.mean()), 6))
print("- violation_rate:", round(float((violations > 0).mean()), 6))


Individual fairness (local) summary
- evaluated pairs: 97695
- max_violation: 0.773675
- avg_violation: 0.022848
- violation_rate: 0.260648


> **Subgroup and Intersection IF**\n> Reuse the pair-level fairness outputs to report within-group IF for marital-status, sex, and the marital-status x sex intersection.


In [14]:
# Within-group IF for selected analysis attributes and their intersection
required = ["rows", "cols", "violations", "X_sensitivefeatures_test"]
missing = [name for name in required if name not in globals()]
if missing:
    raise NameError("Run the local fairness cell first. Missing: " + ", ".join(missing))

sensitive_test = X_sensitivefeatures_test.reset_index(drop=True).copy()
analysis_attrs = ["marital-status", "sex"]


def within_group_if_table_from_labels(labels, attribute_name):
    labels = labels.fillna("<missing>").astype(str)
    labels_arr = labels.to_numpy()
    group_i = labels_arr[rows]
    group_j = labels_arr[cols]
    within_mask = group_i == group_j
    sample_counts = labels.value_counts().to_dict()

    records = []
    for subgroup in sorted(sample_counts):
        mask = within_mask & (group_i == subgroup)
        n_pairs = int(mask.sum())
        if n_pairs == 0:
            records.append({
                "attribute": attribute_name,
                "subgroup": subgroup,
                "n_samples": int(sample_counts[subgroup]),
                "n_pairs": 0,
                "max_violation": np.nan,
                "avg_violation": np.nan,
                "violation_rate": np.nan,
            })
            continue

        v = violations[mask]
        records.append({
            "attribute": attribute_name,
            "subgroup": subgroup,
            "n_samples": int(sample_counts[subgroup]),
            "n_pairs": n_pairs,
            "max_violation": float(v.max()),
            "avg_violation": float(v.mean()),
            "violation_rate": float((v > 0).mean()),
        })

    table = pd.DataFrame(records)
    return table.sort_values(["violation_rate", "n_pairs"], ascending=[False, False], na_position="last")

for attribute in analysis_attrs:
    print(f"\nWithin-group IF by {attribute}")
    print(within_group_if_table_from_labels(sensitive_test[attribute], attribute).to_string(index=False))

intersection_name = "marital-status x sex"
intersection_labels = (
    sensitive_test["marital-status"].fillna("<missing>").astype(str)
    + " | "
    + sensitive_test["sex"].fillna("<missing>").astype(str)
)
print("\nWithin-group IF by (marital-status, sex) intersection")
print(
    within_group_if_table_from_labels(
        intersection_labels,
        intersection_name,
    ).to_string(index=False)
)



Within-group IF by marital-status
     attribute              subgroup  n_samples  n_pairs  max_violation  avg_violation  violation_rate
marital-status              Divorced        883     2640       0.396209       0.024547        0.318561
marital-status    Married-civ-spouse       2977    25074       0.773675       0.028579        0.301508
marital-status         Never-married       2181    18553       0.601545       0.012216        0.152482
marital-status             Separated        199      144       0.403079       0.011038        0.145833
marital-status Married-spouse-absent         86       14       0.038749       0.005536        0.142857
marital-status               Widowed        182      358       0.273141       0.004430        0.069832
marital-status     Married-AF-spouse          5        0            NaN            NaN             NaN

Within-group IF by sex
attribute subgroup  n_samples  n_pairs  max_violation  avg_violation  violation_rate
      sex     Male       4355   